# 04 — Sémantique : ontologie, RDF et SPARQL

Nos tables disent ce que contiennent les colonnes, pas ce que signifient les lignes. L'ontologie écrit ce sens :
un département est situé dans une région, une mesure de jour de froid est une mesure journalière, etc.

1. décrire le domaine avec une **ontologie OWL** (classes, relations, attributs) ;
2. y charger les régions, départements et résultats gold sous forme de **triplets RDF** ;
3. les interroger avec **SPARQL** (3 requêtes utiles au persona) ;
4. relier nos régions à **Wikidata** ;
5. visualiser le **graphe de connaissances**.

In [1]:
%pip install -q rdflib pyvis

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD

DOSSIER = "/home/jovyan/work/ontologie"
EN = Namespace("http://example.org/enedis#")       # l'espace de noms de notre projet
WD = Namespace("http://www.wikidata.org/entity/")  # Wikidata

g = Graph()
g.bind("en", EN); g.bind("owl", OWL); g.bind("wd", WD)

## 1. L'ontologie : le vocabulaire du domaine

On déclare les **classes** (les types de choses), les **relations** entre elles et les **attributs** (les valeurs).
Les sous-classes (⊂) permettent au graphe de faire des déductions : une mesure de jour de froid EST une mesure journalière.

In [3]:
def classe(nom, libelle, parent=None):
    g.add((EN[nom], RDF.type, OWL.Class))
    g.add((EN[nom], RDFS.label, Literal(libelle, lang="fr")))
    if parent:
        g.add((EN[nom], RDFS.subClassOf, EN[parent]))

def relation(nom, domaine, cible, libelle):
    g.add((EN[nom], RDF.type, OWL.ObjectProperty))
    g.add((EN[nom], RDFS.domain, EN[domaine]))
    g.add((EN[nom], RDFS.range, EN[cible]))
    g.add((EN[nom], RDFS.label, Literal(libelle, lang="fr")))

def attribut(nom, domaine, type_xsd, libelle):
    g.add((EN[nom], RDF.type, OWL.DatatypeProperty))
    g.add((EN[nom], RDFS.domain, EN[domaine]))
    g.add((EN[nom], RDFS.range, type_xsd))
    g.add((EN[nom], RDFS.label, Literal(libelle, lang="fr")))

# Classes (et sous-classes)
classe("Region", "région")
classe("Departement", "département")
classe("ProfilClient", "famille de clients")
classe("ProfilResidentiel", "clients résidentiels (RES)", parent="ProfilClient")
classe("ProfilProfessionnel", "clients professionnels (PRO)", parent="ProfilClient")
classe("ProfilEntreprise", "petites entreprises (ENT)", parent="ProfilClient")
classe("Mesure", "mesure de consommation")
classe("MesureJournaliere", "consommation d'une région sur un jour", parent="Mesure")
classe("MesureJourDeFroid", "consommation d'un des 10 jours les plus froids", parent="MesureJournaliere")
classe("MesurePointe", "puissance moyenne à une heure donnée", parent="Mesure")

# Relations entre objets
relation("situeDans", "Departement", "Region", "est situé dans")
relation("concerne", "Mesure", "Region", "concerne la région")
relation("pourProfil", "MesurePointe", "ProfilClient", "pour la famille de clients")

# Attributs (valeurs)
attribut("codeRegion", "Region", XSD.string, "code INSEE de la région")
attribut("nom", "Region", XSD.string, "nom")
attribut("codeInsee", "Departement", XSD.string, "code INSEE du département")
attribut("gradientMWParDegre", "Region", XSD.decimal, "thermosensibilité (MW par °C)")
attribut("gradientWParClient", "Region", XSD.decimal, "thermosensibilité par client (W par °C)")
attribut("rangReactivite", "Region", XSD.integer, "rang de réactivité au froid")
attribut("date", "MesureJournaliere", XSD.date, "date")
attribut("temperatureMoyenne", "MesureJournaliere", XSD.decimal, "température moyenne (°C)")
attribut("puissanceMoyenneMW", "Mesure", XSD.decimal, "puissance moyenne (MW)")
attribut("pointeMW", "MesureJournaliere", XSD.decimal, "pointe de la journée (MW)")
attribut("mois", "MesurePointe", XSD.integer, "mois (1 à 12)")
attribut("heure", "MesurePointe", XSD.integer, "heure (0 à 23)")

print(len(g), "triplets dans l'ontologie")

86 triplets dans l'ontologie


In [4]:
print(g.serialize(format="turtle")[:2500])

@prefix en: <http://example.org/enedis#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

en:MesureJourDeFroid a owl:Class ;
    rdfs:label "consommation d'un des 10 jours les plus froids"@fr ;
    rdfs:subClassOf en:MesureJournaliere .

en:ProfilEntreprise a owl:Class ;
    rdfs:label "petites entreprises (ENT)"@fr ;
    rdfs:subClassOf en:ProfilClient .

en:ProfilProfessionnel a owl:Class ;
    rdfs:label "clients professionnels (PRO)"@fr ;
    rdfs:subClassOf en:ProfilClient .

en:ProfilResidentiel a owl:Class ;
    rdfs:label "clients résidentiels (RES)"@fr ;
    rdfs:subClassOf en:ProfilClient .

en:codeInsee a owl:DatatypeProperty ;
    rdfs:label "code INSEE du département"@fr ;
    rdfs:domain en:Departement ;
    rdfs:range xsd:string .

en:codeRegion a owl:DatatypeProperty ;
    rdfs:label "code INSEE de la région"@fr ;
    rdfs:domain en:Region ;
    rdfs:range xsd:st